In [ ]:
# ============================================================
# NOTEBOOK 3: User Segmentation for Organic Growth
# File: 03_user_segmentation.ipynb
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROCESSED_PATH = "../data/processed/"

# ============================================================
# STEP 1: Load lightweight user summary
# ============================================================
# Logic:
# We do NOT load order_items_tagged.csv here.
# That file is huge.
# For segmentation, we only need one row per user.

user_summary = pd.read_csv(PROCESSED_PATH + "user_organic_summary.csv")

print("User summary loaded successfully")
print("Shape:", user_summary.shape)
print("Unique users:", user_summary["user_id"].nunique())

display(user_summary.head())

# ============================================================
# STEP 2: Safety check required columns
# ============================================================
# Logic:
# Segmentation needs these columns.
# If any are missing, the notebook should stop clearly.

required_cols = [
    "user_id",
    "total_orders",
    "total_items",
    "organic_items",
    "organic_orders"
]

missing_cols = [col for col in required_cols if col not in user_summary.columns]

if missing_cols:
    raise ValueError(f"Missing columns in user_summary: {missing_cols}")

print("\nAll required columns are present.")

# ============================================================
# STEP 3: Recalculate organic share metrics safely
# ============================================================
# Logic:
# Even if these columns already exist, recalculating makes sure
# the values are correct and consistent.

user_summary["organic_item_share"] = (
    user_summary["organic_items"] / user_summary["total_items"]
)

user_summary["organic_order_share"] = (
    user_summary["organic_orders"] / user_summary["total_orders"]
)

print("\nOrganic share metrics created.")
display(user_summary[[
    "user_id",
    "total_orders",
    "total_items",
    "organic_items",
    "organic_orders",
    "organic_item_share",
    "organic_order_share"
]].head())

# ============================================================
# STEP 4: Define high-value condition
# ============================================================
# Logic:
# This condition must match the high-value definition used in the funnel.
#
# High-value organic user means:
# 1. Organic is at least 25% of their total basket
# 2. They bought organic in at least 5 different orders
# 3. They have at least 10 total orders
#
# This prevents us from wrongly calling a user high-value
# just because they bought organic once.

high_value_condition = (
    (user_summary["organic_item_share"] >= 0.25) &
    (user_summary["organic_orders"] >= 5) &
    (user_summary["total_orders"] >= 10)
)

print("\nHigh-value users according to funnel definition:")
print(high_value_condition.sum())

# ============================================================
# STEP 5: Create organic user segments
# ============================================================
# Growth PM logic:
#
# Funnel tells us WHERE users drop.
# Segmentation tells us WHO those users are.
#
# Segment definitions:
# 1. Non-organic buyer:
#    Never bought organic.
#
# 2. Organic light trial user:
#    Bought only 1 organic item.
#
# 3. Organic early repeat user:
#    Bought some organic, but has fewer than 3 organic orders.
#
# 4. Organic low-share repeat user:
#    Buys organic repeatedly but organic basket share is below 15%.
#
# 5. Organic high-value user:
#    Matches strict high-value condition.
#
# 6. Organic loyal but not high-value:
#    Has meaningful organic behavior but has not reached high-value level.

conditions = [
    user_summary["organic_items"] == 0,
    user_summary["organic_items"] < 2,
    user_summary["organic_orders"] < 3,
    user_summary["organic_item_share"] < 0.15,
    high_value_condition
]

choices = [
    "Non-organic buyer",
    "Organic light trial user",
    "Organic early repeat user",
    "Organic low-share repeat user",
    "Organic high-value user"
]

user_summary["organic_segment"] = np.select(
    conditions,
    choices,
    default="Organic loyal but not high-value"
)

print("\nOrganic segments created.")
display(user_summary[[
    "user_id",
    "total_orders",
    "organic_items",
    "organic_orders",
    "organic_item_share",
    "organic_order_share",
    "organic_segment"
]].head())

# ============================================================
# STEP 6: Create segment summary
# ============================================================
# Logic:
# Now we summarize each segment.
#
# This answers:
# How many users are in each segment?
# How active are they?
# How many organic products do they buy?
# How much of their basket is organic?

segment_summary = user_summary.groupby("organic_segment").agg(
    users=("user_id", "nunique"),
    avg_total_orders=("total_orders", "mean"),
    avg_total_items=("total_items", "mean"),
    avg_organic_items=("organic_items", "mean"),
    avg_organic_orders=("organic_orders", "mean"),
    avg_organic_item_share=("organic_item_share", "mean"),
    avg_organic_order_share=("organic_order_share", "mean")
).reset_index()

segment_summary["user_share_pct"] = (
    segment_summary["users"] / segment_summary["users"].sum() * 100
)

segment_summary = segment_summary.sort_values(
    by="users",
    ascending=False
).reset_index(drop=True)

print("\nSegment summary:")
display(segment_summary)

# ============================================================
# STEP 7: Add Growth PM interpretation labels
# ============================================================
# Logic:
# This makes the segment table more business-friendly.
# Each segment gets a suggested growth action.

growth_action_map = {
    "Non-organic buyer": "Drive first organic trial with starter offers and trust messaging.",
    
    "Organic light trial user": "Push second organic purchase using coupon, reminder, or personalized recommendation.",
    
    "Organic early repeat user": "Build habit through reorder nudges and small organic bundles.",
    
    "Organic low-share repeat user": "Increase organic basket share using organic alternatives and cross-sell.",
    
    "Organic loyal but not high-value": "Upgrade to high-value using category expansion, replenishment bundles, and premium organic packs.",
    
    "Organic high-value user": "Protect and monetize through membership, subscription, referrals, and premium bundles."
}

segment_summary["growth_action"] = segment_summary["organic_segment"].map(growth_action_map)

print("\nSegment summary with growth actions:")
display(segment_summary)

# ============================================================
# STEP 8: Identify main opportunity segment
# ============================================================
# Logic:
# Based on the funnel, biggest drop-off was before high-value.
# So the most important target is:
# Organic loyal but not high-value users.
#
# These users are already warm.
# They are easier to upgrade than cold non-organic users.

main_opportunity = segment_summary[
    segment_summary["organic_segment"] == "Organic loyal but not high-value"
]

print("\nMain opportunity segment:")
display(main_opportunity)

if not main_opportunity.empty:
    opportunity_users = int(main_opportunity["users"].iloc[0])
    opportunity_share = float(main_opportunity["user_share_pct"].iloc[0])
    
    print(
        f"\nMain Growth Opportunity: {opportunity_users:,} users "
        f"({opportunity_share:.2f}% of total users) are loyal but not high-value."
    )

# ============================================================
# STEP 9: Save segmented outputs
# ============================================================

user_summary.to_csv(
    PROCESSED_PATH + "user_organic_summary_segmented.csv",
    index=False
)

segment_summary.to_csv(
    PROCESSED_PATH + "organic_segment_summary.csv",
    index=False
)

print("\nSaved files:")
print("1. user_organic_summary_segmented.csv")
print("2. organic_segment_summary.csv")

# ============================================================
# STEP 10: Plot user segment distribution
# ============================================================
# Logic:
# This chart shows which segments dominate the user base.

plt.figure(figsize=(12, 5))
plt.bar(segment_summary["organic_segment"], segment_summary["users"])
plt.xticks(rotation=45, ha="right")
plt.title("Organic User Segment Distribution")
plt.ylabel("Number of Users")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 11: Plot organic item share by segment
# ============================================================
# Logic:
# This chart shows how deeply organic products are present
# in each segment's basket.

plt.figure(figsize=(12, 5))
plt.bar(
    segment_summary["organic_segment"],
    segment_summary["avg_organic_item_share"] * 100
)
plt.xticks(rotation=45, ha="right")
plt.title("Average Organic Item Share by Segment")
plt.ylabel("Average Organic Item Share (%)")
plt.tight_layout()
plt.show()

# ============================================================
# STEP 12: Final written insight
# ============================================================
# Logic:
# This gives you a report-ready interpretation.

print("\n==================== FINAL SEGMENTATION INSIGHT ====================")

print(
    """
The segmentation analysis divides users based on organic purchase depth,
organic order frequency, and organic basket share.

The most important segment for growth is 'Organic loyal but not high-value'.
These users already show meaningful organic buying behavior but do not yet
meet the strict high-value criteria of 25%+ organic basket share, 5+ organic
orders, and 10+ total orders.

This means the main growth opportunity is not simply first-time organic trial.
The stronger opportunity is to convert already-engaged organic users into
high-value organic buyers through basket expansion, organic category cross-sell,
replenishment bundles, and personalized organic alternatives.
"""
)

print("Notebook 3 completed successfully.")